In [ ]:
import os, warnings
import numpy as np
import scipy as sp
from src.data import DataLoader
from src.timescales import Timescales

import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
from src.plotting import plot_autocorrelation_analysis, plot_subject_timeseries_heatmap

warnings.simplefilter("ignore", UserWarning)

In [ ]:
import os
import numpy as np
import pandas as pd
from src.data import DataLoader

# Select the dataset to process.
# Available: 'HCP-YA', 'HCP-D', 'WAND', 'PNC', 'NKI', 'UCLA', 'Penn_LEAD', 'HBN'
dataset = 'HCP-YA'

data_loader = DataLoader(dataset=dataset)

# Load fMRI data

In [ ]:
from src.notebook import DATASET_PARAMS

# ── Load parameters for the selected dataset ─────────────────────────────────
# Edit DATASET_PARAMS in src/notebook.py to change processing parameters.
params = DATASET_PARAMS[dataset]
nuisance_regression = params['nuisance_regression']
task_regression     = params['task_regression']

subject_ids = data_loader.load_subject_ids(params['subject_ids_file'])
# subject_ids = subject_ids[:20]  # limit for testing

print(f"Dataset:    {dataset}")
print(f"Modalities: {params['modalities']}")
print(f"Subjects:   {len(subject_ids)}")
subject_ids[:5]

In [ ]:
for modality in params['modalities']:
    data_loader.register_scans(
        subject_ids=subject_ids,
        sessions=params['sessions'],
        tasks=params['tasks'][modality],
        space=params['space'],
        seg=params['seg'],
        modality=modality,
    )

print(f"Registered {len(data_loader.scan_registry)} scans across {len(params['modalities'])} modality(ies).")

In [ ]:
data_loader.validate_scan_registry(
    verbose=True,
    remove_incomplete_subjects=params['remove_incomplete_subjects'],
)

In [ ]:
timeseries_data = data_loader.load_batch(
    nuisance_regression=nuisance_regression,
    verbose=True,
)

In [ ]:
# The registry filename encodes the dataset and, for HCP-YA, the active modality.
active_modality = params['modalities'][0] if len(params['modalities']) == 1 else 'multi'
modality_suffix = {
    'MRI3T': '',
    'MRI7T': '_7T',
    'MEG':   '_MEG',
    'multi': '_multi',
}.get(active_modality, f'_{active_modality}')

registry_filename = f'scan_registry_{dataset}{modality_suffix}.csv'
data_loader.export_scan_registry(registry_filename)

scan_registry = data_loader.list_registered_scans()
scan_registry.head()

In [ ]:
subject_ids = scan_registry['sub'].unique()
n_subs = len(subject_ids)
print(n_subs)

In [ ]:
subject_ids

## Compute functional connectivity

In [ ]:
# def compute_fc(ts):
#     """
#     Parameters
#     ----------
#     ts : np.array (n_parcels, n_timepoints)
#         time series

#     Returns
#     -------
#     fc : np.array (n_parcels, n_parcels)
#         functional connectivity matrix
#     """

#     fc = np.corrcoef(ts, rowvar=True)
#     np.fill_diagonal(fc, np.nan)
#     fc = np.arctanh(fc)
#     np.fill_diagonal(fc, 1)

#     return fc

In [ ]:
# from tqdm import tqdm
# import json

In [ ]:
# functional_connectivity = dict().fromkeys(timeseries_data)

# for idx in tqdm(scan_registry.index):
#     scan_string = scan_registry.loc[idx, 'scan_string']

#     raw_data = timeseries_data[scan_string]
#     functional_connectivity[scan_string] = compute_fc(raw_data)

In [ ]:
# if dataset == 'HCP-YA':
#     np.savez('/mnt/storage_ssd_raid/research_projects/intrinsic_neural_timescales/data/HCP-YA_fc', **functional_connectivity)

# Process fMRI timescales

In [ ]:
timescales_processor = Timescales(dataset=dataset, timeseries_data=timeseries_data, verbose=True)
if task_regression:
    timescales_processor.regress_task_structure()

### Save out time series for Emily

In [ ]:
run_cell = False

In [ ]:
if run_cell:
    timeseries_data_out = dict()
    for scan_key in timescales_processor.timeseries_data.keys():
        if 'task-rfMRIREST' in scan_key:
            timeseries_data_out[scan_key] = np.array(timescales_processor.timeseries_data[scan_key]).copy()
    print(len(list(timeseries_data_out.keys())))
    np.save(os.path.expanduser('~/Desktop/HCP-YA_Schaefer400-7_task-rest_data-ts.npy'), timeseries_data_out)

    timeseries_data_out = dict()
    for scan_key in timescales_processor.timeseries_data.keys():
        if 'task-tfMRIWM' in scan_key:
            timeseries_data_out[scan_key] = np.array(timescales_processor.timeseries_data[scan_key]).copy()
    print(len(list(timeseries_data_out.keys())))      
    np.save(os.path.expanduser('~/Desktop/HCP-YA_Schaefer400-7_task-wm_data-ts.npy'), timeseries_data_out)

    timeseries_data_out = dict()
    for scan_key in timescales_processor.timeseries_residuals.keys():
        if 'task-tfMRIWM' in scan_key:
            timeseries_data_out[scan_key] = np.array(timescales_processor.timeseries_residuals[scan_key]).copy()
    print(len(list(timeseries_data_out.keys())))  
    np.save(os.path.expanduser('~/Desktop/HCP-YA_Schaefer400-7_task-wm_data-tsresid.npy'), timeseries_data_out)

In [ ]:
timescales_processor.get_min_scan_length_per_subject()

### Timeseries metrics (not timescales)

In [ ]:
run_cell = True

In [ ]:
if run_cell:
    for method in ['tsnr', 'var']:
        timescales_processor.compute_timeseries_measures(method=method, force_recompute=False, verbose=True)

        if task_regression:
            timescales_processor.compute_timeseries_measures(method=method, use_residuals=True, force_recompute=False, verbose=True)

### Timescale measures

In [ ]:
run_cell = True

In [ ]:
timescale_measures = [
    'zero_crossing',
    'zero_crossing_robust_nc2',
    'zero_crossing_robust_nc3',
    'auc_zero_crossing',
    # 'auc_half_crossing',
    # 'decay_time_half',
    # 'lag_one',

    # 'tau',
    # 'tau_howell',
    # 'tau_murray_par',
    # 'B_murray_par',
    # 'B_murray_par_lag3mins',
    # 'B_murray_par_lag4mins',
    # 'B_murray_par_lag5mins',
]

In [ ]:
if run_cell:
    timescales_processor.compute_autocorrelation_function(use_fft=True, use_residuals=False, force_recompute=False, verbose=True)
    for timescale_measure in timescale_measures:
        timescales_processor.compute_timescale_measures(method=timescale_measure, use_residuals=False, force_recompute=False, verbose=True)
        
    autocorr_dict_raw = timescales_processor.autocorr_dict.copy()
    timescales_dict_raw = timescales_processor.timescales_dict.copy()

In [ ]:
if run_cell and task_regression:
    timescales_processor.compute_autocorrelation_function(use_fft=True, use_residuals=True, force_recompute=False, verbose=True)
    for timescale_measure in timescale_measures:
        timescales_processor.compute_timescale_measures(method=timescale_measure, use_residuals=True, suffix='resids', force_recompute=False, verbose=True)
        
    autocorr_dict_resid = timescales_processor.autocorr_dict.copy()
    timescales_dict_resid = timescales_processor.timescales_dict.copy()

# Generate plots

In [ ]:
run_cell = True

In [ ]:
if run_cell:
    for idx in scan_registry.index[:25]:
    # for idx in [2]:
        # if idx & 10 == 0 and scan_registry.loc[idx, 'condition'] == 'task':
        if scan_registry.loc[idx, 'condition'] == 'task':
            scan_string = scan_registry.loc[idx, 'scan_string']
            tr = scan_registry.loc[idx, 'tr']
            print(idx, scan_string)
        
            autocorr_raw = autocorr_dict_raw[scan_string]
            autocorr_resid = autocorr_dict_resid[scan_string]
            timescales_raw = timescales_dict_raw[scan_string]
            timescales_resid = timescales_dict_resid[scan_string]

            plot_dir = os.path.join(timescales_processor.output_dir, 'results', dataset, 'subject_plots', 'plot_autocorrelation_analysis')
            if not os.path.exists(plot_dir):
                os.makedirs(plot_dir)

            fig = plot_autocorrelation_analysis(autocorr_raw, autocorr_resid, timescales_raw, timescales_resid, 
                                                tr, figsize=(15, 4), dpi=100, max_lag=50,
                                                save_path=os.path.join(plot_dir, f'{scan_string}.png')
                                                )
            plt.close('all')